# Lab 30: The Garden-Path Trap

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 4, *The Language Engine*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-30.ipynb)

**What you will do:** time how long a garden-path sentence takes you to read compared with a matched control sentence, log where in the sentence you stumbled, and use a small open language model to show that the exact word which trips you up is also the word the model finds most surprising.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 30 you read sentences such as "The horse raced past the barn fell," which most readers
initially misparse. The confusion arrives at a specific word — here, "fell" — because your
parser had already committed to "raced" as the main verb and must now backtrack and
reanalyse the sentence. Once you see the correct structure ("The horse [that was] raced past
the barn fell"), the sentence is easy.

## Record your results

Time yourself, sentence by sentence, reading sentence (1) from Lab 30 aloud, and a matched
control sentence with the same words in an unambiguous order ("The horse that was raced past
the barn fell"). Note whether you stumbled, paused, or had to re-read, for each one.

In [ ]:
# example data: replace with your own stopwatch timings and stumble notes
readings = pd.DataFrame({
    "sentence": ["garden-path: The horse raced past the barn fell.",
                 "control: The horse that was raced past the barn fell.",
                 "garden-path: The old man the boats.",
                 "control: The elderly people crew the boats."],
    "time_s": [3.4, 2.6, 2.1, 1.9],
    "stumbled": [True, False, True, False],
})
display(readings)

readings.set_index("sentence").time_s.plot.barh(
    color=["firebrick", "grey", "firebrick", "grey"], figsize=(7, 3))
plt.xlabel("seconds to read aloud"); plt.tight_layout(); plt.show()

## Compare

Lab 30 does not report a specific published reading-time figure to compare against — its
evidence is eye-tracking (longer fixations and more "regressions," or backward eye jumps,
at the disambiguating word) and brainwave components (N400, P600), not a single number. The
honest comparison here is within yourself: did the garden-path sentences take longer and
cause more stumbles than their unambiguous controls?

In [ ]:
gp = readings[readings.sentence.str.startswith("garden-path")]
ctrl = readings[readings.sentence.str.startswith("control")]
print(f"Mean time, garden-path: {gp.time_s.mean():.1f} s ({gp.stumbled.mean():.0%} stumbled)")
print(f"Mean time, control:     {ctrl.time_s.mean():.1f} s ({ctrl.stumbled.mean():.0%} stumbled)")
if gp.time_s.mean() > ctrl.time_s.mean():
    print("Slower on the garden-path sentences, matching the reanalysis cost Lab 30 describes.")
else:
    print("No slowdown here -- perhaps these particular sentences no longer surprised you,"
          " or the difference was too small for a stopwatch to catch.")

## The AI side

Lab 30 models reading effort as **surprisal**: roughly, how unlikely a word was given
everything read so far (Hale, 2001). Below, a small open language model, DistilGPT-2, reads
the garden-path sentence one word at a time and reports the surprisal (in bits) of each
word — how many bits of "surprise" the model needed to accommodate it. Watch for a spike at
"fell," the word where your own parser had to backtrack.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

def word_surprisals(sentence):
    ids = tokenizer(sentence, return_tensors="pt").input_ids
    with torch.no_grad():
        log_probs = torch.log_softmax(model(ids).logits, dim=-1)
    # surprisal of each token, given every token before it
    token_bits = [-log_probs[0, i - 1, ids[0, i]].item() / np.log(2) for i in range(1, ids.shape[1])]
    tokens = [tokenizer.decode([t.item()]).strip() for t in ids[0, 1:]]
    return pd.DataFrame({"token": tokens, "surprisal_bits": token_bits})

garden_path = word_surprisals("The horse raced past the barn fell.")
control = word_surprisals("The horse that was raced past the barn fell.")

display(garden_path)
garden_path.plot.bar(x="token", y="surprisal_bits", color="firebrick", legend=False, figsize=(8, 3))
plt.ylabel("surprisal (bits)"); plt.title("DistilGPT-2's surprise, word by word"); plt.xticks(rotation=45)
plt.show()
print(f"Peak surprisal in the garden-path sentence: {garden_path.surprisal_bits.max():.1f} bits, "
      f"at '{garden_path.loc[garden_path.surprisal_bits.idxmax(), 'token']}'")

If the peak lands on "fell" (or the space before it), the model is showing the same
computational signature you experienced: a word that violates its running prediction costs
more to process. DistilGPT-2 has no eyes, no reanalysis, and nothing it experiences as
confusion — it is only a table of next-token probabilities. A genuine spike here shows the
statistical pattern behind garden-path sentences is detectable from text alone, not that the
model "understands" the sentence the way you do.

## Pool the class data

Pool reading times and stumble reports across the class, one row per person per sentence
type (anonymous ID, never a name), and check whether the group as a whole is slower on
garden-path sentences.

In [ ]:
import io
csv_text = """id,sentence_type,time_s,stumbled
P01,garden-path,3.6,True
P01,control,2.5,False
P02,garden-path,3.0,True
P02,control,2.8,False
P03,garden-path,2.4,False
P03,control,2.3,False
P04,garden-path,4.1,True
P04,control,2.9,False
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
diffs = (df[df.sentence_type == "garden-path"].set_index("id").time_s
         - df[df.sentence_type == "control"].set_index("id").time_s)

boot = [diffs.sample(len(diffs), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean slowdown on garden-path sentences: {diffs.mean():.1f} s "
      f"(95% bootstrap interval {lo:.1f} to {hi:.1f} s, n = {len(diffs)})")
diffs.plot.hist(bins=6, color="firebrick", alpha=0.7, figsize=(6, 3))
plt.xlabel("garden-path time minus control time (s)"); plt.show()

## Questions to think about

1. The surprisal peak in the AI Lab depends on which words came before it. Try feeding word_surprisals a sentence that removes "past the barn" ("The horse raced fell.") -- does the surprisal at "fell" go up or down, and how does that fit with the idea that the parser needs to have already committed to a reading of "raced" as the main verb?
2. The book describes two separate brainwave signatures for garden-path sentences — the N400 (semantic surprise) and the P600 (syntactic reanalysis). Surprisal, as computed here, is a single number. What aspect of the human response might a single surprisal score be unable to capture?
3. Try the same word_surprisals function on "Time flies like an arrow; fruit flies like a banana." Where is the model most surprised, and does it match where you personally slowed down?
4. The chapter argues that beam search is not, in practice, what stops modern chatbots from generating garden-path-like garbled text -- giving the model room to draft and revise is. Based on what you saw here, why would generating a full sentence in one incremental left-to-right pass make early commitments especially costly to undo?

## Challenge

Write your own two-sentence pair: one genuinely ambiguous garden-path sentence, and a matched
control that says almost the same thing without the ambiguity. Time yourself reading both
aloud, then run both through the word_surprisals function above and see whether the model's
peak lines up with your own point of confusion.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-30.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")